In [267]:
# ** Uncomment to Retrain the model **
# import torch

# **Define Device**
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# !python modules/train.py

In [268]:
import torch
import numpy as np
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)


In [ ]:
import torch
from modules.unet3d import UNet3D  # Ensure `unet3d.py` is in `modules/`

# Define Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_file = '02' # Test File (change as needed)

# Model
model_path = f"models/brain_shift_model_{test_file}.pt"
model = UNet3D().to(device)
'''model.load_state_dict(torch.load(model_path, map_location=device))'''
model.eval()  # Set model to evaluation mode

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [270]:
from modules.ground_truth import parse_tag_file

import os
import sys
sys.path.append(os.path.abspath("modules"))
import numpy as np

# Load landmarks from tag file
tag_file_path = f"Test/{test_file}/landmarks.tag"  
pre_landmarks, post_landmarks = parse_tag_file(tag_file_path)

# Compute ground-truth brain shift distances
landmark_shifts = np.linalg.norm(pre_landmarks - post_landmarks, axis=1)
mean_shift_tag = np.mean(landmark_shifts)
max_shift_tag = np.max(landmark_shifts)

print(f"Ground-Truth Mean Brain Shift (Tag File): {mean_shift_tag:.3f} mm")
print(f"Ground-Truth Max Brain Shift (Tag File): {max_shift_tag:.3f} mm")


Ground-Truth Mean Brain Shift (Tag File): 2.492 mm
Ground-Truth Max Brain Shift (Tag File): 5.417 mm


In [271]:
landmark_shifts

array([1.18813596, 2.40736779, 5.41700074, 0.82717932, 1.70115224,
       4.55137621, 3.37788259, 2.15296114, 0.80049695])

In [272]:
print("Shape of landmark_shifts:", landmark_shifts.shape)
print("First few values:", landmark_shifts[:5])


Shape of landmark_shifts: (9,)
First few values: [1.18813596 2.40736779 5.41700074 0.82717932 1.70115224]


In [273]:
from modules.load_model import load_brain_shift_model
from modules.dataset import UltrasoundDataset
from torch.utils.data import DataLoader

# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
"""model = load_brain_shift_model("models/brain_shift_model.pt", device)
"""

# Load Dataset
dataset = UltrasoundDataset(f"Test")
train_loader = DataLoader(dataset, batch_size=1, shuffle=False)

# Store shift values
all_shifts = []



In [274]:


from modules.test import compute_brain_shift, world_to_voxel
from modules.dataset import load_mnc
import numpy as np
import nibabel as nib

# Load the image
img = nib.load(f"Test/{test_file}/pre.mnc")


predicted_landmark_shifts = []  # Store model-predicted shifts at landmark locations

# Image spacing
spacing = np.array([0.3, 0.3, 0.3])  # Corrected voxel spacing

# Assuming origin is (0,0,0) unless otherwise specified in the dataset
origin = img.affine[:3, 3] 


In [275]:
# Get original image shape BEFORE resizing
original_img_shape = nib.load(f"Test/{test_file}/pre.mnc").shape  # Shape BEFORE resizing

# Get the resizing scale factor
scale_factor = np.array(original_img_shape) / np.array([128, 128, 128])  

# Rescale Voxel Landmarks
pre_landmarks_voxel = (pre_landmarks - origin) / spacing
pre_landmarks_voxel = pre_landmarks_voxel / scale_factor  # Scale to match (128,128,128)
pre_landmarks_voxel = np.round(pre_landmarks_voxel).astype(int)  

# Rescaled landmarks
print(f"Rescaled Voxel Landmarks:\n{pre_landmarks_voxel}")


Rescaled Voxel Landmarks:
[[ 50  81 105]
 [ 50 109 110]
 [ 74  83  71]
 [ 64  46  95]
 [ 64  62  96]
 [ 42  81  63]
 [ 50  82  71]
 [ 56  52  63]
 [ 61  52 102]]


In [276]:
# Ensure voxel coordinates fit within (0,127)
valid_landmarks = []
for x, y, z in pre_landmarks_voxel.reshape(-1, 3):
    if (0 <= x < 128) and (0 <= y < 128) and (0 <= z < 128):  
        valid_landmarks.append((x, y, z))
    else:
        print(f"Skipping out-of-bounds landmark: {x, y, z}")

valid_landmarks = np.array(valid_landmarks)

print(f"Valid Voxel Landmarks: {valid_landmarks.shape}")
print(valid_landmarks)


Valid Voxel Landmarks: (9, 3)
[[ 50  81 105]
 [ 50 109 110]
 [ 74  83  71]
 [ 64  46  95]
 [ 64  62  96]
 [ 42  81  63]
 [ 50  82  71]
 [ 56  52  63]
 [ 61  52 102]]


In [277]:
import numpy as np

predicted_landmark_shifts = []  # Store predicted shift values

# Run Inference
model.eval()
for pre, post, pre_landmarks, post_landmarks in train_loader:
    pre, post = pre.to(device), post.to(device)
    with torch.no_grad():
        flow = model(pre, post)
        mean_shift, max_shift, shift_map = compute_brain_shift(flow)

    print(f"Mean Brain Shift: {mean_shift:.3f} mm")
    print(f"Max Brain Shift: {max_shift:.3f} mm")

    # Extract shifts at valid landmark positions
    for x, y, z in valid_landmarks:  
        predicted_landmark_shifts.append(shift_map[0, :, x, y, z])  

predicted_landmark_shifts = np.array(predicted_landmark_shifts)

# Convert ground-truth shifts to 3D vectors
ground_truth_vectors = (post_landmarks - pre_landmarks).cpu().numpy().squeeze(0)

print(f"Shape of predicted landmark shifts: {predicted_landmark_shifts.shape}")  # (9, 3)
print(f"Shape of corrected ground-truth shifts: {ground_truth_vectors.shape}")  # (9, 3)


Shift Map Output Shape: torch.Size([1, 3, 128, 128, 128])
Mean Brain Shift: 3.152 mm
Max Brain Shift: 3.201 mm
Shape of predicted landmark shifts: (9, 3)
Shape of corrected ground-truth shifts: (9, 3)


In [278]:
# Step 1: Build a mask of valid indices
valid_mask = []
valid_voxels = []
for i, (x, y, z) in enumerate(pre_landmarks_voxel.reshape(-1, 3)):
    if (0 <= x < 128) and (0 <= y < 128) and (0 <= z < 128):  
        valid_mask.append(i)
        valid_voxels.append((x, y, z))

# Step 2: Apply mask to GT and Pred
ground_truth_vectors = ground_truth_vectors[valid_mask]
predicted_landmark_shifts = np.array([shift_map[0, :, x, y, z] for x, y, z in valid_voxels])


In [ ]:
import numpy as np
from scipy.stats import iqr

# Convert 3D vectors to shift magnitudes
predicted_magnitudes = np.linalg.norm(predicted_landmark_shifts, axis=1)  # (9,)
ground_truth_magnitudes = np.linalg.norm(ground_truth_vectors, axis=1)  # (9,)

# normalize vectors to a unit vector
gt_norm = ground_truth_vectors / (np.linalg.norm(ground_truth_vectors, axis=1, keepdims=True) + 1e-6)
pred_norm = predicted_landmark_shifts / (np.linalg.norm(predicted_landmark_shifts, axis=1, keepdims=True) + 1e-6)

print(f'for {test_file} landmarks:')

# Mean Absolute Error (MAE)
mae = np.mean(np.abs(predicted_magnitudes - ground_truth_magnitudes))
mae_sd = np.std(np.abs(predicted_magnitudes - ground_truth_magnitudes))
print(f"Mean Absolute Error (MAE): {mae:.3f} ± {mae_sd:.3f} mm")

# Median Absolute Error
median_ae = np.median(np.abs(predicted_magnitudes - ground_truth_magnitudes))
mae_iqr = iqr(np.abs(predicted_magnitudes - ground_truth_magnitudes))
print(f"Median Absolute Error (MedAE): {median_ae:.3f} mm")
print(f"MAE IQR: {mae_iqr:.3f} mm")

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(np.mean((predicted_magnitudes - ground_truth_magnitudes) ** 2))
print(f"Root Mean Squared Error (RMSE): {rmse:.3f} mm")

# Cosine Similarity
cosine_sim = np.sum(gt_norm * pred_norm, axis=1)  # Dot product
cosine_sim = np.clip(cosine_sim, -1.0, 1.0)
angular_error = np.arccos(cosine_sim) * (180 / np.pi)
mean_cosine_sim = np.mean(cosine_sim)
print(f"Mean Cosine Similarity: {mean_cosine_sim:.3f}")

# Mean Angular Error
mean_angular_error = np.mean(angular_error)
angular_sd = np.std(angular_error)
print(f"Mean Angular Error: {mean_angular_error:.2f}° ± {angular_sd:.2f}°")

# Median Angular Error
median_angular_error = np.median(angular_error)
angular_iqr = iqr(angular_error)

print(f"Median Angular Error: {median_angular_error:.2f}°")
print(f"Angular Error IQR: {angular_iqr:.2f}°")


Mean Absolute Error (MAE): 1.525 ± 0.719 mm
Median Absolute Error (MedAE): 1.450 mm
MAE IQR: 1.272 mm
Root Mean Squared Error (RMSE): 1.686 mm
Mean Cosine Similarity: -0.225
Mean Angular Error: 104.99° ± 31.20°
Median Angular Error: 112.80°
Angular Error IQR: 54.28°


In [280]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Extract X, Y, Z coordinates of valid landmarks
x, y, z = valid_landmarks[:, 0], valid_landmarks[:, 1], valid_landmarks[:, 2]

# Ground Truth Shift Vectors
gt_u, gt_v, gt_w = ground_truth_vectors[:, 0], ground_truth_vectors[:, 1], ground_truth_vectors[:, 2]

# Predicted Shift Vectors
pred_u, pred_v, pred_w = predicted_landmark_shifts[:, 0], predicted_landmark_shifts[:, 1], predicted_landmark_shifts[:, 2]



In [281]:
x, y, z

(array([50, 50, 74, 64, 64, 42, 50, 56, 61]),
 array([ 81, 109,  83,  46,  62,  81,  82,  52,  52]),
 array([105, 110,  71,  95,  96,  63,  71,  63, 102]))

In [282]:
gt_u, gt_v, gt_w

(array([ 0.       , -0.6264343, -5.323349 ,  0.       , -0.5676117,
        -1.6691589, -1.4186707, -1.5025787,  0.5645294], dtype=float32),
 array([-0.9917221 , -2.0816917 ,  0.15892792, -0.6502838 , -0.39886475,
        -4.1367264 , -2.8630753 , -1.222496  , -0.56588745], dtype=float32),
 array([-0.654335  , -1.034195  , -0.99024963,  0.51123047,  1.5532684 ,
         0.90356445, -1.0955658 , -0.939682  , -0.04328156], dtype=float32))

In [283]:
pred_u, pred_v, pred_w

(array([2.3227212, 2.326212 , 2.3176734, 2.3245242, 2.3291435, 2.3327827,
        2.3320966, 2.3384805, 2.3220084], dtype=float32),
 array([0.06130997, 0.06476286, 0.06018246, 0.06103801, 0.05962572,
        0.05834654, 0.05660336, 0.05431435, 0.06564116], dtype=float32),
 array([2.1212845, 2.1269999, 2.1082308, 2.1070547, 2.1215627, 2.1180754,
        2.1296415, 2.130901 , 2.1192498], dtype=float32))

In [284]:
import os
import numpy as np
import pandas as pd

output_folder = "brainshiftvectors"
os.makedirs(output_folder, exist_ok=True)

# Dictionary with all the values
data = {
    "X": valid_landmarks[:, 0],
    "Y": valid_landmarks[:, 1],
    "Z": valid_landmarks[:, 2],
    "GT_U": ground_truth_vectors[:, 0],
    "GT_V": ground_truth_vectors[:, 1],
    "GT_W": ground_truth_vectors[:, 2],
    "Pred_U": predicted_landmark_shifts[:, 0],
    "Pred_V": predicted_landmark_shifts[:, 1],
    "Pred_W": predicted_landmark_shifts[:, 2],
}

df = pd.DataFrame(data)

# Save in the specified folder
csv_filename = os.path.join(output_folder, f"brain_shift_vectors_test{test_file}.csv")
df.to_csv(csv_filename, index=False)

print(f"✅ Data saved to {csv_filename}")


✅ Data saved to brainshiftvectors\brain_shift_vectors_test02.csv


In [285]:
data

{'X': array([50, 50, 74, 64, 64, 42, 50, 56, 61]),
 'Y': array([ 81, 109,  83,  46,  62,  81,  82,  52,  52]),
 'Z': array([105, 110,  71,  95,  96,  63,  71,  63, 102]),
 'GT_U': array([ 0.       , -0.6264343, -5.323349 ,  0.       , -0.5676117,
        -1.6691589, -1.4186707, -1.5025787,  0.5645294], dtype=float32),
 'GT_V': array([-0.9917221 , -2.0816917 ,  0.15892792, -0.6502838 , -0.39886475,
        -4.1367264 , -2.8630753 , -1.222496  , -0.56588745], dtype=float32),
 'GT_W': array([-0.654335  , -1.034195  , -0.99024963,  0.51123047,  1.5532684 ,
         0.90356445, -1.0955658 , -0.939682  , -0.04328156], dtype=float32),
 'Pred_U': array([2.3227212, 2.326212 , 2.3176734, 2.3245242, 2.3291435, 2.3327827,
        2.3320966, 2.3384805, 2.3220084], dtype=float32),
 'Pred_V': array([0.06130997, 0.06476286, 0.06018246, 0.06103801, 0.05962572,
        0.05834654, 0.05660336, 0.05431435, 0.06564116], dtype=float32),
 'Pred_W': array([2.1212845, 2.1269999, 2.1082308, 2.1070547, 2.1215627